
# Thí nghiệm 02 — Scenario 4: Train/Test Evaluation

## 1. Chuẩn bị dữ liệu CMIP6 Wind Speed

### 1.1. Mục tiêu

Notebook chuẩn bị bộ dữ liệu tốc độ gió CMIP6 phục vụ thực nghiệm đánh giá mô hình trên tập Train và Test độc lập theo thời gian.

Quy trình bao gồm:

1. Đọc dữ liệu tốc độ gió từ NetCDF.
2. Trích xuất chuỗi thời gian tại 5 vị trí đã xác định.
3. Lựa chọn 2920 thời điểm đầu tiên.
4. Chia dữ liệu theo thời gian với tỷ lệ 80% Train và 20% Test.
5. Chuẩn hóa dữ liệu bằng Min-Max scaler chỉ được khớp trên Train.
6. Kiểm tra các cửa sổ trượt.
7. Lưu dữ liệu và cấu hình thực nghiệm để sử dụng trong notebook Training.

### 1.2. Nguồn dữ liệu

Tệp nguồn:

`datasets/climate/raw/Klima_Daten_10m_3h_2020_RCP26.nc`

Biến sử dụng:

`sfcWind`

Đây là tốc độ gió gần bề mặt thuộc dữ liệu mô phỏng khí hậu, với tần suất ba giờ.

Notebook sử dụng dữ liệu của 5 vị trí không gian tương ứng với các thí nghiệm CMIP6 trước đó.

### 1.3. Xác định vị trí không gian

Vị trí được đọc từ:

`data/pressure_nearest_5sites.pt`

Các trường được sử dụng:

- `grid_rows`: chỉ số hàng trên lưới.
- `grid_cols`: chỉ số cột trên lưới.
- `location_indices`: chỉ số của 5 vị trí đã chọn.

Các ô lưới đã xác định:

| Vị trí | Grid row | Grid column |
|---|---:|---:|
| 0 | 187 | 184 |
| 5000 | 246 | 202 |
| 10000 | 234 | 209 |
| 15000 | 222 | 216 |
| 20000 | 210 | 225 |

Việc tái sử dụng các chỉ số này giúp giữ nguyên vị trí không gian giữa những thí nghiệm CMIP6 Wind và Surface Pressure.

Notebook không lựa chọn lại các vị trí.

### 1.4. Trích xuất chuỗi thời gian

Từ dữ liệu NetCDF, notebook lựa chọn 2920 thời điểm đầu tiên và 5 ô lưới đã xác định.

Kết quả:

$$
W_{\mathrm{raw}}\in\mathbb{R}^{2920\times5}
$$

Trong đó:

- 2920 là số thời điểm.
- 5 là số kênh tốc độ gió.
- Mỗi hàng tương ứng một thời điểm.
- Mỗi cột tương ứng một vị trí không gian.

Dữ liệu được chuyển sang kiểu `float32`.

### 1.5. Chia dữ liệu theo thời gian

Notebook sử dụng tỷ lệ:

$$
80\%\ \mathrm{Train},\qquad20\%\ \mathrm{Test}
$$

Vị trí chia:

$$
N_{\mathrm{train}}=\lfloor0.8\times2920\rfloor=2336
$$

Kích thước dữ liệu:

$$
W_{\mathrm{train}}\in\mathbb{R}^{2336\times5}
$$

$$
W_{\mathrm{test}}\in\mathbb{R}^{584\times5}
$$

Tập Train lấy các thời điểm đầu, tập Test lấy các thời điểm cuối.

Không thực hiện xáo trộn thứ tự thời gian.

### 1.6. Dữ liệu trung gian

Tệp đầu ra của bước trích xuất:

`data/wind_2920_5sites.pt`

Tệp chứa:

- Chuỗi tốc độ gió gốc.
- Thông tin 5 vị trí không gian.
- Chỉ số chia Train/Test.
- Tên tệp NetCDF nguồn.

Dữ liệu đã được tạo trong lần thực nghiệm trước và được giữ nguyên trong quá trình tái cấu trúc repository.


In [ ]:
from pathlib import Path

import numpy as np
import torch
import xarray as xr


# 1. Xác định thư mục gốc của project
project_root = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "experiments" / "02_climate").is_dir()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục gốc của project."
    )


# 2. Xác định đường dẫn dữ liệu
experiment_dir = project_root / "experiments" / "02_climate"
raw_dir = project_root / "datasets" / "climate" / "raw"

locations_file = (
    experiment_dir / "data" / "pressure_nearest_5sites.pt"
)

wind_path = (
    raw_dir / "Klima_Daten_10m_3h_2020_RCP26.nc"
)


# 3. Kiểm tra các tệp đầu vào
if not locations_file.is_file():
    raise FileNotFoundError(locations_file)

if not wind_path.is_file():
    raise FileNotFoundError(wind_path)


# 4. Đọc thông tin vị trí từ dataset Pressure đã chuẩn bị
locations = torch.load(
    locations_file,
    map_location="cpu",
    weights_only=True
)

rows = locations["grid_rows"]
cols = locations["grid_cols"]
location_indices = locations["location_indices"]


# 5. Trích xuất 2920 thời điểm tốc độ gió tại 5 ô lưới
with xr.open_dataset(wind_path) as ds:
    wind_selected = ds["sfcWind"].isel(
        time=slice(0, 2920),
        rlat=xr.DataArray(rows, dims="site"),
        rlon=xr.DataArray(cols, dims="site")
    )

    wind_raw = (
        wind_selected
        .transpose("time", "site")
        .values.astype(np.float32)
    )


# 6. Kiểm tra dữ liệu đã trích xuất
assert wind_raw.shape == (2920, 5)
assert np.isfinite(wind_raw).all()


# 7. Xác định điểm chia Train/Test theo thời gian
split_index = int(0.8 * len(wind_raw))


# 8. Lưu dữ liệu gốc và thông tin chia tập
output_path = (
    experiment_dir / "data" / "wind_2920_5sites.pt"
)

torch.save({
    "wind_raw": torch.from_numpy(wind_raw),
    "location_indices": location_indices,
    "grid_rows": rows,
    "grid_cols": cols,
    "train_split_index": split_index,
    "source": wind_path.name
}, output_path)


# 9. Tổng hợp thông tin dữ liệu
print("=== SCENARIO 4: CMIP6 WIND SPEED ===")

print("Raw:", wind_raw.shape)
print("Train:", wind_raw[:split_index].shape)
print("Test:", wind_raw[split_index:].shape)

print("Locations:", location_indices)
print("Grid rows:", rows)
print("Grid cols:", cols)

print("Split index:", split_index)
print("Saved:", output_path)

c:\Users\Win 11 New\Desktop\pa gk\.venv\lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()



## 2. Chia Train/Test, chuẩn hóa và tạo cửa sổ trượt

### 2.1. Mục tiêu

Sau khi trích xuất dữ liệu tốc độ gió, notebook chuẩn bị hai tập Train và Test để đánh giá khả năng tổng quát hóa của các mô hình.

Khác với các thí nghiệm trước chỉ đánh giá trên dữ liệu huấn luyện, Scenario 4 sử dụng một đoạn thời gian riêng cho kiểm tra.

### 2.2. Chia dữ liệu theo thời gian

Dữ liệu được chia theo thứ tự xuất hiện:

$$
W_{\mathrm{train}}=W[0:2336]
$$

$$
W_{\mathrm{test}}=W[2336:2920]
$$

Kích thước:

| Tập | Số thời điểm | Số kênh |
|---|---:|---:|
| Train | 2336 | 5 |
| Test | 584 | 5 |

Không thực hiện xáo trộn dữ liệu.

Việc chia theo thời gian giúp mô hình được huấn luyện trên đoạn dữ liệu trước và đánh giá trên đoạn dữ liệu sau.

### 2.3. Chuẩn hóa Min-Max

Notebook sử dụng `MinMaxScaler` để chuẩn hóa từng kênh.

Công thức:

$$
x'=\frac{x-x_{\min}}{x_{\max}-x_{\min}}
$$

Các giá trị $x_{\min}$ và $x_{\max}$ được xác định chỉ từ tập Train.

Quy trình:

1. Khởi tạo `MinMaxScaler`.
2. Sử dụng `fit_transform()` trên tập Train.
3. Sử dụng `transform()` trên tập Test bằng scaler đã khớp ở bước trước.

Cách triển khai này không sử dụng các thống kê của Test để xây dựng phép chuẩn hóa.

### 2.4. Phạm vi giá trị sau chuẩn hóa

Từ output đã lưu:

| Tập | Giá trị nhỏ nhất | Giá trị lớn nhất |
|---|---:|---:|
| Train | 0.0 | 1.0 |
| Test | -0.002704557 | 1.1441792 |

Test có giá trị nằm ngoài đoạn [0, 1] vì phép chuẩn hóa sử dụng giới hạn được học từ Train.

Điều này không phải lỗi.

Đặc biệt, không được clip Test về [0, 1], vì thao tác đó sẽ làm thay đổi các đầu vào và nhãn đã sử dụng khi huấn luyện và đánh giá.

### 2.5. Tạo cửa sổ trượt

Với chuỗi dữ liệu $D$, độ dài đầu vào $L$ và chân trời dự báo $H$:

$$
X_i=D[i:i+L]
$$

$$
Y_i=D[i+L:i+L+H]
$$

Nhãn được chuyển vị rồi làm phẳng theo thứ tự kênh và bước dự báo.

Với 5 kênh:

$$
X_i\in\mathbb{R}^{120\times5}
$$

$$
Y_i\in\mathbb{R}^{5H}
$$

Notebook hiện tạo cửa sổ theo:

`range(len(data) - seq_len - horizon)`

Số cửa sổ tương ứng:

$$
N=T-L-H
$$

Phải giữ nguyên công thức này để duy trì tính tương thích với kết quả đã lưu.

### 2.6. Ví dụ với Horizon = 10

Khi:

$$
L=120,\qquad H=10
$$

Tập Train có:

$$
2336-120-10=2206
$$

cửa sổ.

Tập Test có:

$$
584-120-10=454
$$

cửa sổ.

Kích thước tensor:

| Tensor | Shape |
|---|---|
| `x_train` | `(2206, 120, 5)` |
| `y_train` | `(2206, 50)` |
| `x_test` | `(454, 120, 5)` |
| `y_test` | `(454, 50)` |

Các cửa sổ của Train và Test được tạo riêng biệt, không có cửa sổ nào đi qua ranh giới giữa hai tập.

### 2.7. Không gian Grid Search

Độ dài chuỗi đầu vào cố định:

$$
L=120
$$

Patch lengths:

$$
P\in\{40,50,60,70,80,90,100\}
$$

Forecast horizons:

$$
H\in\{2,4,6,8,10,12,14,16\}
$$

Số tổ hợp:

$$
7\times8=56
$$

Với bốn mô hình:

$$
56\times4=224
$$

Các cấu hình sử dụng:

- `d_model = 48`.
- `epochs = 4000`.
- `learning_rate = 3e-4`.
- `lyap_weight = 0.1` đối với các mô hình Koopman.
- `seed = 7`.

### 2.8. Nguyên tắc bảo toàn

Không thay đổi tỷ lệ chia Train/Test, phạm vi chuẩn hóa hoặc công thức tạo cửa sổ.

Việc sửa các thành phần này sẽ tạo bộ dữ liệu khác với bộ dữ liệu đã sử dụng trong thực nghiệm 224 lượt huấn luyện.


In [ ]:
from sklearn.preprocessing import MinMaxScaler


# 1. Chia Train/Test theo thứ tự thời gian
train_raw = wind_raw[:split_index]
test_raw = wind_raw[split_index:]


# 2. Fit Min-Max scaler chỉ trên tập Train
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train_raw)
test_scaled = scaler.transform(test_raw)


# 3. Hàm tạo cửa sổ trượt cho một tập dữ liệu
def build_windows(data, seq_len, horizon):
    X, Y = [], []

    for i in range(len(data) - seq_len - horizon):
        X.append(data[i:i + seq_len])

        Y.append(
            data[i + seq_len:i + seq_len + horizon]
            .T.reshape(-1)
        )

    x_in = torch.tensor(
        np.stack(X),
        dtype=torch.float32
    )

    x_out = torch.tensor(
        np.stack(Y),
        dtype=torch.float32
    )

    return x_in, x_out


# 4. Cấu hình thực nghiệm Scenario 4
seq_len = 120

patch_lengths = [40, 50, 60, 70, 80, 90, 100]
horizons = [2, 4, 6, 8, 10, 12, 14, 16]

d_model = 48
epochs = 4000
learning_rate = 3e-4
lyap_weight = 0.1
seed = 7


# 5. Kiểm tra tạo cửa sổ với Horizon = 10
example_horizon = 10

x_train, y_train = build_windows(
    train_scaled,
    seq_len,
    example_horizon
)

x_test, y_test = build_windows(
    test_scaled,
    seq_len,
    example_horizon
)


# 6. Xác minh kích thước các tensor
assert x_train.shape == (2206, 120, 5)
assert y_train.shape == (2206, 50)

assert x_test.shape == (454, 120, 5)
assert y_test.shape == (454, 50)


# 7. Tổng hợp dữ liệu Train/Test
print("=== SCENARIO 4: TRAIN/TEST DATASET ===")

print("Raw:", wind_raw.shape)
print("Train raw:", train_raw.shape)
print("Test raw:", test_raw.shape)

print("\nAfter sliding windows (H=10):")

print("X_train:", x_train.shape)
print("Y_train:", y_train.shape)

print("X_test:", x_test.shape)
print("Y_test:", y_test.shape)


# 8. Kiểm tra phạm vi giá trị sau chuẩn hóa
print("\nScaling:")

print(
    "Train range:",
    train_scaled.min(),
    train_scaled.max()
)

print(
    "Test range:",
    test_scaled.min(),
    test_scaled.max()
)


# 9. Tổng hợp không gian Grid Search
print("\nGrid Search:")

print("Input sequence length:", seq_len)
print("Patch lengths:", patch_lengths)
print("Horizons:", horizons)
print("d_model:", d_model)
print("Epochs:", epochs)

print(
    "Configurations:",
    len(patch_lengths) * len(horizons)
)

=== SCENARIO 4: TRAIN/TEST DATASET ===
Raw: (2920, 5)
Train raw: (2336, 5)
Test raw: (584, 5)

After sliding windows (H=10):
X_train: torch.Size([2206, 120, 5])
Y_train: torch.Size([2206, 50])
X_test: torch.Size([454, 120, 5])
Y_test: torch.Size([454, 50])

Scaling:
Train range: 0.0 1.0
Test range: -0.002704557 1.1441792

Grid Search:
Input sequence length: 120
Patch lengths: [40, 50, 60, 70, 80, 90, 100]
Horizons: [2, 4, 6, 8, 10, 12, 14, 16]
d_model: 48
Epochs: 4000
Configurations: 56



## 3. Lưu dữ liệu và cấu hình Scenario 4

### 3.1. Mục tiêu

Sau khi trích xuất, chia tập và chuẩn hóa, notebook lưu dữ liệu phục vụ thực nghiệm vào một tệp PyTorch duy nhất.

Tệp này được sử dụng trực tiếp bởi:

`09_climate_train_test_training.ipynb`

Nhờ đó, notebook Training không cần đọc lại tệp NetCDF hoặc thực hiện lại phép chia Train/Test và Min-Max scaling.

### 3.2. Dữ liệu được lưu

Tệp:

`data/scenario4_wind_train_test.pt`

Bao gồm dữ liệu Train và Test trước và sau chuẩn hóa.

| Trường | Kích thước | Nội dung |
|---|---|---|
| `train_raw` | `(2336, 5)` | Dữ liệu Train gốc |
| `test_raw` | `(584, 5)` | Dữ liệu Test gốc |
| `train_scaled` | `(2336, 5)` | Dữ liệu Train đã chuẩn hóa |
| `test_scaled` | `(584, 5)` | Dữ liệu Test đã chuẩn hóa |

Các tensor được lưu dưới định dạng `float32`.

### 3.3. Thông tin Min-Max scaler

Notebook lưu các giá trị:

- `scaler_min`: giá trị nhỏ nhất của từng kênh trên Train.
- `scaler_max`: giá trị lớn nhất của từng kênh trên Train.

Hai trường này cho phép truy xuất khoảng giá trị của dữ liệu Train được sử dụng khi chuẩn hóa.

Chúng không phải dữ liệu được tính từ Test.

### 3.4. Cấu hình Grid Search

Tệp còn lưu:

| Trường | Giá trị |
|---|---|
| `seq_len` | 120 |
| `patch_lengths` | `[40, 50, 60, 70, 80, 90, 100]` |
| `horizons` | `[2, 4, 6, 8, 10, 12, 14, 16]` |
| `d_model` | 48 |
| `epochs` | 4000 |
| `learning_rate` | 0.0003 |
| `lyap_weight` | 0.1 |
| `seed` | 7 |
| `train_fraction` | 0.8 |

Đồng thời lưu danh sách 5 vị trí không gian qua trường `location_indices`.

### 3.5. Tổng số lượt huấn luyện

Grid Search gồm:

$$
7\times8=56
$$

tổ hợp patch length và forecast horizon.

Với bốn mô hình:

$$
56\times4=224
$$

lượt huấn luyện.

Mỗi lượt sử dụng 4000 epoch.

### 3.6. Liên hệ với notebook Training

File `09_climate_train_test_training.ipynb` đọc trực tiếp tệp Scenario 4 đã chuẩn bị.

Notebook Training sử dụng:

- `train_scaled` để xây dựng các cửa sổ huấn luyện.
- `test_scaled` để xây dựng các cửa sổ kiểm tra.
- Các siêu tham số đã lưu để xác định cấu hình mô hình.

Nhờ vậy, các kết quả huấn luyện và đánh giá đều dựa trên cùng một phép chia dữ liệu.

### 3.7. Bảo toàn kết quả

Bộ dữ liệu Scenario 4 đã được tạo từ lần thực nghiệm trước.

Trong quá trình tái cấu trúc:

- Không chạy lại NetCDF extraction.
- Không thay đổi điểm chia Train/Test.
- Không khớp lại Min-Max scaler.
- Không thay đổi danh sách cấu hình.
- Không ghi đè tệp `scenario4_wind_train_test.pt`.

Notebook được giữ lại để tài liệu hóa đầy đủ quy trình chuẩn bị dữ liệu.


In [ ]:
# 1. Xác định đường dẫn lưu dữ liệu Scenario 4
output_dir = experiment_dir / "data"
output_dir.mkdir(parents=True, exist_ok=True)

save_path = output_dir / "scenario4_wind_train_test.pt"


# 2. Đóng gói dữ liệu và cấu hình thực nghiệm
dataset_bundle = {
    # Dữ liệu sau chuẩn hóa
    "train_scaled": torch.tensor(
        train_scaled, dtype=torch.float32
    ),
    "test_scaled": torch.tensor(
        test_scaled, dtype=torch.float32
    ),

    # Dữ liệu gốc
    "train_raw": torch.tensor(
        train_raw, dtype=torch.float32
    ),
    "test_raw": torch.tensor(
        test_raw, dtype=torch.float32
    ),

    # Thông tin vị trí và Min-Max scaler
    "location_indices": list(location_indices),
    "scaler_min": torch.tensor(
        scaler.data_min_, dtype=torch.float32
    ),
    "scaler_max": torch.tensor(
        scaler.data_max_, dtype=torch.float32
    ),

    # Cấu hình thực nghiệm
    "seq_len": seq_len,
    "patch_lengths": patch_lengths,
    "horizons": horizons,
    "d_model": d_model,
    "epochs": epochs,
    "learning_rate": learning_rate,
    "lyap_weight": lyap_weight,
    "seed": seed,
    "train_fraction": 0.8,
}


# 3. Lưu dữ liệu đã chuẩn bị
torch.save(dataset_bundle, save_path)


# 4. Đọc lại tệp để xác minh cấu trúc
saved = torch.load(
    save_path,
    map_location="cpu",
    weights_only=True
)


# 5. Kiểm tra các thông tin quan trọng
assert saved["train_scaled"].shape == (2336, 5)
assert saved["test_scaled"].shape == (584, 5)

assert saved["seq_len"] == 120
assert saved["epochs"] == 4000


# 6. Tổng hợp kết quả
print("=== SCENARIO 4 DATASET SAVED ===")

print("Train:", saved["train_scaled"].shape)
print("Test:", saved["test_scaled"].shape)

print("Input sequence length:", saved["seq_len"])
print("Patch lengths:", saved["patch_lengths"])
print("Horizons:", saved["horizons"])
print("Epochs:", saved["epochs"])

print(
    "Total training runs:",
    len(patch_lengths) * len(horizons) * 4
)

print(
    "File size (MB):",
    round(save_path.stat().st_size / 1024**2, 3)
)

print("Saved:", save_path)

=== SCENARIO 4 DATASET SAVED ===
Train: torch.Size([2336, 5])
Test: torch.Size([584, 5])
Input sequence length: 120
Patch lengths: [40, 50, 60, 70, 80, 90, 100]
Horizons: [2, 4, 6, 8, 10, 12, 14, 16]
Epochs: 4000
Total training runs: 224
File size (MB): 0.114
Saved: c:\Users\Win 11 New\Desktop\pa gk\experiments\02_climate\data\scenario4_wind_train_test.pt
